# Item-item co-purchase pairs

**What this notebook does.** It reads the interaction log and writes every
unordered pair of items the same user bought within `WINDOW_DAYS` of each
other to `data/co_purchase_pairs.pkl`, using only interactions from before the
shared train/test cutoff date in `ttn/constants.json`.

**What it is used for.** This is the co-purchase signal read straight off the
reviews — which items real users actually bought together. Its sibling
`categories.ipynb` builds the same idea from Amazon's `also_buy` lists at the
*category* level; this one stays at the item level and owes nothing to what
Amazon claims. Two different signals, and they need not agree.

**The two filters, and why each is there.**

- **The cutoff.** Interactions at or after the split point are dropped before
  any pairing happens, so a table built for training cannot encode what
  happened during the evaluation window. Strictly-before, the same convention
  as `compute_cooccurrence_before_time` in `functions/rs_baseline_models.py`,
  `InteractionMatrixBuilder`, and the split in `ttn/ttn_complementary.ipynb` §6.
- **The window.** A pair counts only where the two purchases sit
  within `WINDOW_DAYS` of each other. A user who bought a blender in 2019 and
  a duvet in 2023 was not shopping for a set; without a window every item a
  long-lived account ever bought pairs with every other one. The test is on the
  gap itself, not on recency, so a tightly spaced 2014 pair still counts.

Repeat purchases of the same item are kept rather than collapsed, so a pair
counts if **any** event of A and **any** event of B fall inside the window.

Every function lives in `pairs.py` — this notebook defines no logic of its own:

```python
df_reviews  = load_interactions(INTERACTIONS_PATH)
cutoff_time = pd.Timestamp(DATE_THRESHOLD).timestamp()   # ttn/constants.json
pairs       = co_purchase_pairs(df_reviews, cutoff_time=cutoff_time,
                                window_days=WINDOW_DAYS)
save_co_purchase_pairs(pairs, OUT_PATH)
```


In [ ]:
import json
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from complementary_cats_pairs import (
    co_purchase_pairs,
    load_interactions,
    save_co_purchase_pairs,
)

DATA_DIR = PROJECT_ROOT / "data"
INTERACTIONS_PATH = DATA_DIR / "Home_and_Kitchen_filtered.csv"
OUT_PATH = DATA_DIR / "co_purchase_pairs.pkl"

# The train/test split point is shared with ttn/ttn_complementary.ipynb §6, which reads this
# same file. It is written down in exactly one place so the two cannot drift.
CONSTANTS_PATH = PROJECT_ROOT / "ttn" / "constants.json"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the interaction log

`Home_and_Kitchen_filtered.csv` is one row per review, i.e. one row per
(user, item, date) event. Only the three columns the pairing needs are read —
the file is 851 MB across 11 columns, and the other eight are dead weight here.

Rows are **not** de-duplicated on load. Repeat purchases on different days are
separate events and can each anchor a window; only exact duplicate
`(user, item, timestamp)` triples are dropped, inside `co_purchase_pairs`,
because they can only repeat pairs another row already makes.


In [ ]:
if not INTERACTIONS_PATH.exists():
    raise FileNotFoundError(
        f"{INTERACTIONS_PATH} not found. It is the filtered review log — see "
        "data/variable_selection.ipynb, which builds it from the raw dump."
    )

df_reviews = load_interactions(INTERACTIONS_PATH)

span = pd.to_datetime(df_reviews["unixReviewTime"], unit="s")
print(f"interactions : {len(df_reviews):,}")
print(f"unique users : {df_reviews['reviewerID'].nunique():,}")
print(f"unique items : {df_reviews['asin'].nunique():,}")
print(f"date range   : {span.min().date()} .. {span.max().date()}")
display(df_reviews.head(5))


## 2. The cutoff and the window

The split point is **not** written here. It lives in `ttn/constants.json` as
`date_threshold`, and `ttn/ttn_complementary.ipynb` §6 reads the same key for its train/test
split. One value, in one file, read by both notebooks — so the split the
two-tower model trains on and the split these pairs are built from are the same
by construction rather than by convention. Edit the JSON and both follow; it is
git-tracked, so the change is a one-line diff someone can review.

Interactions **strictly before** the threshold are paired; everything at or
after it is held out. One global cutoff rather than a per-user one, which is
what keeps every training row ahead of every test row.

`WINDOW_DAYS` is the maximum gap between the two purchases of a pair, and is
local to this notebook — `ttn_complementary.ipynb` has no use for it. Change either that or
the shared date and re-run; nothing above this cell depends on them.


In [ ]:
constants = json.loads(CONSTANTS_PATH.read_text())
DATE_THRESHOLD = constants["date_threshold"]   # shared with ttn/ttn_complementary.ipynb §6

WINDOW_DAYS = 90    # maximum gap between the two purchases of a pair

# Naive timestamps are treated as UTC, which is the basis unixReviewTime is on.
cutoff_time = pd.Timestamp(DATE_THRESHOLD).timestamp()

before = int((df_reviews["unixReviewTime"] < cutoff_time).sum())

print(f"cutoff : {DATE_THRESHOLD} (unix {int(cutoff_time):,}), strictly before")
print(f"         from {CONSTANTS_PATH.relative_to(PROJECT_ROOT)}")
print(f"kept   : {before:,} of {len(df_reviews):,} interactions "
      f"({before / len(df_reviews):.1%})")
print(f"window : {WINDOW_DAYS} days between the two purchases")

## 3. Build the pairs and save

The two filters are reported separately below before the combined figure, which
is the quickest way to see which of the two is doing the cutting — the same
reading `categories.ipynb` §3 gives for support and lift.

`save_co_purchase_pairs` keeps both columns as `category` dtype, so each side
is an integer code against one shared asin vocabulary rather than a repeated
10-character string. Re-running this cell overwrites the output.


In [ ]:
pairs = co_purchase_pairs(
    df_reviews, cutoff_time=cutoff_time, window_days=WINDOW_DAYS,
)
save_co_purchase_pairs(pairs, OUT_PATH)

# What each filter costs on its own, for comparison against the combined run.
unfiltered = co_purchase_pairs(df_reviews, cutoff_time=None, window_days=10**5)
window_only = co_purchase_pairs(df_reviews, cutoff_time=None,
                                window_days=WINDOW_DAYS)
cutoff_only = co_purchase_pairs(df_reviews, cutoff_time=cutoff_time,
                                window_days=10**5)

n_all = len(unfiltered)
print(f"no filters      : {n_all:>12,} pairs")
print(f"window alone    : {len(window_only):>12,} pairs "
      f"({len(window_only) / n_all:.1%} of unfiltered)")
print(f"cutoff alone    : {len(cutoff_only):>12,} pairs "
      f"({len(cutoff_only) / n_all:.1%} of unfiltered)")
print(f"both            : {len(pairs):>12,} pairs "
      f"({len(pairs) / n_all:.1%} of unfiltered)")
print(f"items involved  : "
      f"{pd.concat([pairs['asinA'], pairs['asinB']]).nunique():>12,}")
print(f"in memory       : {pairs.memory_usage(deep=True).sum() / 1e6:>12,.0f} MB")
print(f"\nsaved -> {OUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUT_PATH.stat().st_size / 1e6:.2f} MB)")

del unfiltered, window_only, cutoff_only
display(pairs.head(10))


---

Read the result back with:

```python
pairs = pd.read_pickle(DATA_DIR / "co_purchase_pairs.pkl")
```

To rebuild from the log in one call, skipping this notebook:

```python
from complementary_cats_pairs import run_co_purchase_pairs

pairs = run_co_purchase_pairs(
    interactions=DATA_DIR / "Home_and_Kitchen_filtered.csv",
    cutoff_time=pd.Timestamp(DATE_THRESHOLD).timestamp(),
    window_days=90,
    out_path=DATA_DIR / "co_purchase_pairs.pkl",
)
```

`data/*.pkl` is gitignored, so the output stays local like every other
generated table in this project.
